In [3]:
import pandas as pd

In [5]:
df=pd.read_csv(r"C:\Users\moudy\Downloads\20237012\20237012\Q2_clean_event_log.csv")

In [7]:
df.head()

,case_id,activity,timestamp,resource
0,C001,File Claim,2026-04-03 08:17:00,Dave
1,C001,Assess Liability,2026-04-03 08:32:00,Jana
2,C001,Inspect Damage,2026-04-03 09:16:00,Laila
3,C001,Approve Payout,2026-04-03 09:46:00,Erin
4,C001,Request Evidence,2026-04-03 11:37:00,Jana


In [11]:
import pandas as pd
from itertools import combinations, product


df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(["case_id", "timestamp"])  # add a stable tiebreak if timestamps tie

acts = sorted(df.activity.unique())
lab = {a: chr(97+i) for i, a in enumerate(acts)}
print(lab)

traces = df.groupby("case_id")["activity"].apply(lambda s: tuple(lab[a] for a in s))
L = set(traces)
print("Q2.1 variants:", len(L))

# directly-follows
DF = {(t[i], t[i+1]) for t in L for i in range(len(t)-1)}
print("Q2.2 |>L|:", len(DF))

TL = sorted(lab.values())
def rel(x, y):
    xy, yx = (x, y) in DF, (y, x) in DF
    return "->" if xy and not yx else "<-" if yx and not xy else "||" if xy and yx else "#"

pairs = list(combinations(TL, 2))
R = {p: rel(*p) for p in pairs}
print("Q2.3 causality:", sum(r in ("->", "<-") for r in R.values()))
print("Q2.4 parallel:", sum(r == "||" for r in R.values()))
print("Q2.5 #:", sum(r == "#" for r in R.values()))
for p in pairs: print(p, R[p])          # Q2.14

TI = sorted({t[0] for t in L}); TO = sorted({t[-1] for t in L})
print("Q2.6/7", len(TI), ",".join(TI)); print("Q2.8/9", len(TO), ",".join(TO))

caus = lambda x, y: rel(x, y) == "->"
indep = lambda S: all(rel(x, y) == "#" for x in S for y in S)  # includes x#x
def subsets(S):
    for r in range(1, len(S)+1):
        yield from combinations(S, r)
XL = [(set(A), set(B)) for A in subsets(TL) for B in subsets(TL)
      if indep(A) and indep(B) and all(caus(a, b) for a in A for b in B)]
print("Q2.10 |XL|:", len(XL))
YL = [(A, B) for A, B in XL
      if not any((A <= A2 and B <= B2) and (A, B) != (A2, B2) for A2, B2 in XL)]
print("Q2.11 |YL|:", len(YL))
print("Q2.12 places:", len(YL) + 2)
FL = sum(len(A) + len(B) for A, B in YL) + len(TI) + len(TO)
print("Q2.13 arcs:", FL)

{'Approve Payout': 'a', 'Assess Liability': 'b', 'Assign Adjuster': 'c', 'Close Claim': 'd', 'File Claim': 'e', 'Inspect Damage': 'f', 'Notify Claimant': 'g', 'Request Evidence': 'h'}
Q2.1 variants: 12
Q2.2 |>L|: 17
Q2.3 causality: 9
Q2.4 parallel: 4
Q2.5 #: 15
('a', 'b') <-
('a', 'c') #
('a', 'd') #
('a', 'e') #
('a', 'f') ||
('a', 'g') ->
('a', 'h') ||
('b', 'c') <-
('b', 'd') #
('b', 'e') <-
('b', 'f') ||
('b', 'g') #
('b', 'h') ||
('c', 'd') #
('c', 'e') #
('c', 'f') ->
('c', 'g') #
('c', 'h') #
('d', 'e') #
('d', 'f') #
('d', 'g') <-
('d', 'h') #
('e', 'f') ->
('e', 'g') #
('e', 'h') #
('f', 'g') #
('f', 'h') ->
('g', 'h') <-
Q2.6/7 2 c,e
Q2.8/9 1 d
Q2.10 |XL|: 11
Q2.11 |YL|: 7
Q2.12 places: 9
Q2.13 arcs: 19


In [14]:
df = df.sort_values(["case_id", "timestamp"])
print(df.groupby("case_id")["activity"].apply(tuple).nunique())

12


In [16]:
df = df.sort_values(["case_id", "timestamp"])
pairs = set()
for _, g in df.groupby("case_id"):
    acts = list(g["activity"])
    pairs |= set(zip(acts, acts[1:]))
print(len(pairs))

17


In [18]:
df = df.sort_values(["case_id", "timestamp"])
DF = set()
for _, g in df.groupby("case_id"):
    a = list(g["activity"])
    DF |= set(zip(a, a[1:]))

acts = sorted(df["activity"].unique())
causal = [(x, y) for x, y in combinations(acts, 2)
          if ((x, y) in DF) != ((y, x) in DF)]   # exactly one direction
print(len(causal))

9


In [20]:
df = df.sort_values(["case_id", "timestamp"])
DF = set()
for _, g in df.groupby("case_id"):
    a = list(g["activity"])
    DF |= set(zip(a, a[1:]))

acts = sorted(df["activity"].unique())
parallel = [(x, y) for x, y in combinations(acts, 2)
            if (x, y) in DF and (y, x) in DF]   # both directions occur
print(len(parallel))

4


In [22]:
df = df.sort_values(["case_id", "timestamp"])
DF = set()
for _, g in df.groupby("case_id"):
    a = list(g["activity"])
    DF |= set(zip(a, a[1:]))

acts = sorted(df["activity"].unique())
unrelated = [(x, y) for x, y in combinations(acts, 2)
             if (x, y) not in DF and (y, x) not in DF]   # neither direction occurs
print(len(unrelated))

15


In [24]:
df = df.sort_values(["case_id", "timestamp"])
TI = df.groupby("case_id")["activity"].first().unique()
print(len(TI), sorted(TI))

2 ['Assign Adjuster', 'File Claim']


In [26]:
df = df.sort_values(["case_id", "timestamp"])
lab = {a: chr(97+i) for i, a in enumerate(sorted(df.activity.unique()))}
TI = sorted({lab[a] for a in df.groupby("case_id")["activity"].first()})
print(",".join(TI))

c,e


In [28]:
df = df.sort_values(["case_id", "timestamp"])
lab = {a: chr(97+i) for i, a in enumerate(sorted(df.activity.unique()))}
TO = sorted({lab[a] for a in df.groupby("case_id")["activity"].last()})
print(len(TO), ",".join(TO))

1 d


In [30]:
df = df.sort_values(["case_id", "timestamp"])
lab = {a: chr(97+i) for i, a in enumerate(sorted(df.activity.unique()))}
TO = sorted({lab[a] for a in df.groupby("case_id")["activity"].last()})
print(",".join(TO))

d


In [32]:
df = df.sort_values(["case_id", "timestamp"])
lab = {a: chr(97+i) for i, a in enumerate(sorted(df.activity.unique()))}
DF = set()
for _, g in df.groupby("case_id"):
    a = [lab[x] for x in g["activity"]]
    DF |= set(zip(a, a[1:]))

TL = sorted(lab.values())
caus  = lambda x, y: (x, y) in DF and (y, x) not in DF
indep = lambda S: all((x, y) not in DF and (y, x) not in DF for x in S for y in S)  # includes x#x

def subsets(S):
    for r in range(1, len(S) + 1):
        yield from combinations(S, r)

XL = [(A, B) for A in subsets(TL) for B in subsets(TL)
      if indep(A) and indep(B) and all(caus(a, b) for a in A for b in B)]
print(len(XL))

11


In [34]:

df = df.sort_values(["case_id", "timestamp"])
lab = {a: chr(97+i) for i, a in enumerate(sorted(df.activity.unique()))}
DF = set()
for _, g in df.groupby("case_id"):
    a = [lab[x] for x in g["activity"]]
    DF |= set(zip(a, a[1:]))

TL = sorted(lab.values())
caus  = lambda x, y: (x, y) in DF and (y, x) not in DF
indep = lambda S: all((x, y) not in DF and (y, x) not in DF for x in S for y in S)

def subsets(S):
    for r in range(1, len(S) + 1):
        yield from combinations(S, r)

XL = [(set(A), set(B)) for A in subsets(TL) for B in subsets(TL)
      if indep(A) and indep(B) and all(caus(a, b) for a in A for b in B)]

YL = [(A, B) for A, B in XL
      if not any(A <= A2 and B <= B2 and (A, B) != (A2, B2) for A2, B2 in XL)]
print(len(YL))

7


In [36]:

df = df.sort_values(["case_id", "timestamp"])
lab = {a: chr(97+i) for i, a in enumerate(sorted(df.activity.unique()))}
DF = set()
for _, g in df.groupby("case_id"):
    a = [lab[x] for x in g["activity"]]
    DF |= set(zip(a, a[1:]))

TL = sorted(lab.values())
caus  = lambda x, y: (x, y) in DF and (y, x) not in DF
indep = lambda S: all((x, y) not in DF and (y, x) not in DF for x in S for y in S)

def subsets(S):
    for r in range(1, len(S) + 1):
        yield from combinations(S, r)

XL = [(set(A), set(B)) for A in subsets(TL) for B in subsets(TL)
      if indep(A) and indep(B) and all(caus(a, b) for a in A for b in B)]
YL = [(A, B) for A, B in XL
      if not any(A <= A2 and B <= B2 and (A, B) != (A2, B2) for A2, B2 in XL)]

print(len(YL) + 2)

9


In [38]:

df = df.sort_values(["case_id", "timestamp"])
lab = {a: chr(97+i) for i, a in enumerate(sorted(df.activity.unique()))}
DF = set()
for _, g in df.groupby("case_id"):
    a = [lab[x] for x in g["activity"]]
    DF |= set(zip(a, a[1:]))

TL = sorted(lab.values())
caus  = lambda x, y: (x, y) in DF and (y, x) not in DF
indep = lambda S: all((x, y) not in DF and (y, x) not in DF for x in S for y in S)

def subsets(S):
    for r in range(1, len(S) + 1):
        yield from combinations(S, r)

XL = [(set(A), set(B)) for A in subsets(TL) for B in subsets(TL)
      if indep(A) and indep(B) and all(caus(a, b) for a in A for b in B)]
YL = [(A, B) for A, B in XL
      if not any(A <= A2 and B <= B2 and (A, B) != (A2, B2) for A2, B2 in XL)]

TI = {lab[a] for a in df.groupby("case_id")["activity"].first()}
TO = {lab[a] for a in df.groupby("case_id")["activity"].last()}

print(sum(len(A) + len(B) for A, B in YL) + len(TI) + len(TO))

19
